# MNIST Klassifikation
Wir wollen untersuchen, wie stabil (oder umgekehrt „sensibel“) die Klassifikation von MNIST-
Bildern mit unseren KNNs ist. Konkret schauen wir uns an, ob Änderungen jeweils einzelner
Bildpunkte dazu führen können, dass ein ursprünglich korrektes Ergebnis falsch wird.
Führen Sie dazu folgendes Experiment durch:

## gutes MNIST
- Trainieren Sie ein Netz für MNIST das gut funktioniert
- Speichern Sie das Netz mit ```torch.save(model.state_dict(), ...)```

In [88]:
from time import time
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42) # Reproduzierbarkeit

# Data Class
class CFG:
    data_dir: str = "./data"
    batch_size: int = 256
    epochs: int = 4
    lr: float = 1e-3
    weight_decay: float = 1e-4
    model_path: str = "mnist_cnn_state_dict.pt"

    device: str = (
        "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    )

cfg = CFG()

### Model

In [96]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        #self.conv1 = nn.Conv2d(1, 32, kernel_size = 3, padding=1)   # 1x28x28 -> 32x28x28
        #self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, padding=1)   # 32x14x14 -> 64x14x14
        #self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.conv1 = nn.Conv2d(1, 16, kernel_size = 3, padding=1)   # 1x28x28 -> 16x28x28
        self.conv2 = nn.Conv2d(16, 32, kernel_size = 3, padding=1)   # 16x14x14 -> 32x14x14
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)      # 32x14x14
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)      # 64x7x7
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)             # Logits
        return x 

### Data

In [90]:
def get_loaders():
    transform = transforms.Compose([
        transforms.ToTensor(),                          # [0,1]
        #transforms.Normalize((0.1307,), (0.3081,)),     # Standard-MNIST-Norm
    ])
    train_ds = datasets.MNIST(cfg.data_dir, train=True,  download=True, transform=transform)
    test_ds  = datasets.MNIST(cfg.data_dir, train=False, download=True, transform=transform)

    # Für MPS/CPU: pin_memory=False; workers ruhig >0 (macOS kann das)
    num_workers = 0
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=num_workers, pin_memory=False, persistent_workers=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,num_workers=num_workers, pin_memory=False, persistent_workers=False)

    return train_loader, test_loader

### Train/Eval

In [91]:
def train_one_epoch(model, loader, opt, criterion, device):
    model.train()
    total_loss = 0.0
    n = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        opt.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)

    return total_loss / n

In [92]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return correct / total

In [93]:
def fit():
    train_loader, test_loader = get_loaders()
    model = Net().to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.CrossEntropyLoss()

    acc0 = evaluate(model, test_loader, cfg.device)
    print(f"Test-Acc vor Training: {round(acc0*100, 2)}%") # sollte ca 10% sein

    best_acc = 0.0
    t0 = time()
    for epoch in range(1, cfg.epochs + 1):
        t_last = time()
        tr_loss = train_one_epoch(model, train_loader, opt, criterion, cfg.device)
        te_acc = evaluate(model, test_loader, cfg.device)

        print(f"Epoch {epoch}/{cfg.epochs}\t | loss={tr_loss:.4f}\t | test_acc={te_acc*100:.2f}%\t | epoch_dur: {time()-t_last:.1f}s")

        if te_acc > best_acc:
            best_acc = te_acc
            torch.save(model.state_dict(), cfg.model_path)

    print(f"Beste Test-Acc: {best_acc*100:.2f}%\t | Dauer: {time()-t0:.1f}s")

    return model

### Run Model

In [94]:
# Trainieren und bestes state_dict unter data/ speicher
model = fit()

Test-Acc vor Training: 9.84%
Epoch 1/4	 | loss=0.4614	 | test_acc=95.87%	 | epoch_dur: 6.0s
Epoch 2/4	 | loss=0.1086	 | test_acc=97.80%	 | epoch_dur: 5.2s
Epoch 3/4	 | loss=0.0692	 | test_acc=98.43%	 | epoch_dur: 5.1s
Epoch 4/4	 | loss=0.0535	 | test_acc=98.51%	 | epoch_dur: 5.1s
Beste Test-Acc: 98.51%	 | Dauer: 21.4s


## Klassifizierungstest
- Laden Sie das Netz in einem neuen Projekt mit ```model.load_state_dict(torch.load(...))```
- Laden Sie die MNIST-Trainingsdaten
- Nutzen Sie einen `DataLoder`, um alle Trainings-Bilder einzeln nacheinander verarbeiten zu können

- Für jedes Bild bestimmen Sie zunächst die Klasse, der ihr Modell das Bild zuordnet
- Wenn das Bild korrekt klassifiziert wird: Ändern Sie nacheinander jeden einzelnen Pixel, indem Sie z.B. den Wert invertieren: $ f(a) = 1 - a $
- Kontrollieren Sie nach jeder einzelnen (der 784 möglichen, unabhänbgigen) Veränderung, ob ihr Modell das Bild immer noch richtig klassifiziert

Bereiten Sie Ihre Ergebnisse auf, z.B.
- Wie viele der Trainings-Bilder sind "angreifbar", welche sind "robust" in dem Sinne, dass keine einzige der Veränderungen zu einer Falsch-Klassifikation führt?
- Welche Vertauschungen zwischen Klassen können Sie wie häufig provozieren?
- Auf die Veränderung welcher Bildpunkte reagiert Ihr Modell besonders häufig?
- (Sie können diese Auswertungen auch auf eine einzige Klasse beschränken, dann müssen Sie nur ca. 1/10 der Bilder verarbeiten und alles geht schneller)